# 📊 Pipeline de Preparação de Dados — Passos Mágicos (2022-2024)

## 📝 Objetivo do Notebook

Este notebook implementa o pipeline de **preparação e consolidação** de dados educacionais da Associação Passos Mágicos, cobrindo os anos de **2022, 2023 e 2024**.

O objetivo é produzir um dataset estruturado e limpo para modelagem preditiva de risco de defasagem escolar.

---

## 🎯 Lógica e Decisões de Processamento

### 1. Desafios Identificados

| Desafio | Descrição | Impacto |
|---------|-----------|---------|
| **Heterogeneidade de Colunas** | Cada ano tem colunas diferentes e nomenclaturas distintas | Alto |
| **Valores Ausentes** | Alto índice de nulos em features comportamentais | Médio |
| **Inconsistência de Fase** | '9' é valor inválido em 2022/2023, mas válido em 2024 | Alto |
| **Divergência de Nomenclatura** | 'Defas' vs 'DEFASAGEM', 'Data de Nasc' vs 'DATA_DE_NASC' | Médio |

### 2. Estratégia de Consolidação

```mermaid
flowchart TD
    A[Carregar dados brutos por ano] --> B[Padronizar nomenclatura]
    B --> C[Adicionar colunas ausentes]
    C --> D[Transformar FASE e TURMA]
    D --> E[Calcular features derivadas]
    E --> F[Consolidar em dataset único]
    F --> G[Análise de qualidade final]
```

### 3. Transformações Principais

#### 3.1 Padronização de Colunas
- **Regra**: `{nome_original}_{ano_curto}` em MAIÚSCULAS
- **Exemplo**: `Nome Aluno` (2022) → `NOME_ALUNO_22`

#### 3.2 Normalização de FASE

| Original | Normalizado (2022/2023) | Normalizado (2024) |
|----------|------------------------|-------------------|
| ALFA | Fase 0 (1° e 2° ano) | Fase 0 (1° e 2° ano) |
| 8A/8B | Fase 8 (Universitários) | Fase 8 (Universitários) |
| **9** | ❌ NÃO SE APLICA (descartado) | ✅ Fase 4 (9° ano) |

#### 3.3 Criação de Flags Derivadas
- **VETERANO**: aluno com `ANO_INGRESSO < 2022`
- **EM_FASE**: aluno com `FASE_ATUAL == FASE_IDEAL`

### 4. Garantias de Qualidade

✅ **Preservação de Dados**: Todas as funções retornam cópias (não alteram DataFrames originais)  
✅ **Testabilidade**: Funções isoladas em `scripts/` com testes unitários em `tests/`  
✅ **Rastreabilidade**: Sufixos de ano garantem identificação da origem  
✅ **Reprodutibilidade**: Pipeline determinístico e documentado  

### 5. Métricas de Qualidade Esperadas

| Métrica | Target |
|---------|--------|
| % de nulos por coluna | < 30% |
| Alunos únicos (ID) | ~700-800 |
| Colunas consolidadas | ~80-100 |
| Features derivadas | 10+ |

---

## 🚀 Execução do Pipeline

Este notebook implementa o pipeline usando funções modularizadas em:
- `scripts/data_processing.py` — ETL e limpeza
- `scripts/notebook_feature_engineering.py` — Transformações de domínio

**Referências:**
- Testes: `tests/test_data_processing.py`, `tests/test_notebook_feature_engineering.py`
- Documentação: [README.md](../README.md)


In [ ]:
# Imports necessários
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path

# Adiciona diretórios ao path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Imports de módulos do projeto
from scripts.data_processing import (
    padronizar_colunas_ano,
    adicionar_colunas_vazias,
    analise_nulos,
    calcular_idade_2023,
    obter_elementos_comuns,
    filtrar_colunas_relevantes,
    consolidar_dataframes,
)

from scripts.notebook_feature_engineering import (
    obter_nova_turma,
    obter_nova_fase,
    obter_nova_fase_24,
    criar_coluna_veterano,
    criar_coluna_em_fase,
    aplicar_transformacoes_fase_turma,
)

# Configurações
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✅ Imports realizados com sucesso")
print(f"📁 Project root: {project_root}")

---

## 📂 Etapa 1: Carregamento de Dados Brutos

Carrega arquivos Excel de cada ano.

In [ ]:
# Define caminhos dos arquivos
data_dir = project_root / "app" / "data" / "raw"
arquivo_2022 = data_dir / "BASE DE DADOS PEDE 2022 - DATATHON.xlsx"
arquivo_2023 = data_dir / "BASE DE DADOS PEDE 2023 - DATATHON.xlsx"
arquivo_2024 = data_dir / "BASE DE DADOS PEDE 2024 - DATATHON.xlsx"

# Carrega dados
df_2022 = pd.read_excel(arquivo_2022)
df_2023 = pd.read_excel(arquivo_2023)
df_2024 = pd.read_excel(arquivo_2024)

print(f"📊 Dimensões originais:")
print(f"  2022: {df_2022.shape[0]:>4} linhas × {df_2022.shape[1]:>3} colunas")
print(f"  2023: {df_2023.shape[0]:>4} linhas × {df_2023.shape[1]:>3} colunas")
print(f"  2024: {df_2024.shape[0]:>4} linhas × {df_2024.shape[1]:>3} colunas")

---

## 🔧 Etapa 2: Padronização de Nomenclatura

Aplica padrão `{NOME_COLUNA}_{ANO}` em maiúsculas para todas as colunas.

In [ ]:
# Padroniza colunas por ano
df_2022_pad = padronizar_colunas_ano(df_2022, 2022, ignorar_cols=["NOME"])
df_2023_pad = padronizar_colunas_ano(df_2023, 2023, ignorar_cols=["NOME"])
df_2024_pad = padronizar_colunas_ano(df_2024, 2024, ignorar_cols=["NOME"])

print("✅ Colunas padronizadas")
print("\n📋 Exemplo de colunas 2022 (primeiras 10):")
print(list(df_2022_pad.columns[:10]))

---

## 🧩 Etapa 3: Uniformização de Estrutura

Adiciona colunas ausentes com valores NaN para garantir mesma estrutura entre anos.

In [ ]:
# Define conjunto de colunas esperadas (união de todas)
todas_colunas = set(df_2022_pad.columns) | set(df_2023_pad.columns) | set(df_2024_pad.columns)

# Adiciona colunas faltantes em cada DataFrame
df_2022_uni = adicionar_colunas_vazias(df_2022_pad, list(todas_colunas))
df_2023_uni = adicionar_colunas_vazias(df_2023_pad, list(todas_colunas))
df_2024_uni = adicionar_colunas_vazias(df_2024_pad, list(todas_colunas))

print(f"✅ Estrutura uniformizada: {len(todas_colunas)} colunas totais")
print(f"  2022: {df_2022_uni.shape[1]} colunas (adicionadas {df_2022_uni.shape[1] - df_2022_pad.shape[1]})")
print(f"  2023: {df_2023_uni.shape[1]} colunas (adicionadas {df_2023_uni.shape[1] - df_2023_pad.shape[1]})")
print(f"  2024: {df_2024_uni.shape[1]} colunas (adicionadas {df_2024_uni.shape[1] - df_2024_pad.shape[1]})")

---

## 🎭 Etapa 4: Transformação de FASE e TURMA

Normaliza códigos de fase (ALFA, 8A, etc) para formato padrão.

**Atenção:** Regras de 2024 diferem de 2022/2023 (fase '9').

In [ ]:
# Identifica coluna de fase em cada ano
col_fase_2022 = "FASE_22" if "FASE_22" in df_2022_uni.columns else None
col_fase_2023 = "FASE_23" if "FASE_23" in df_2023_uni.columns else None
col_fase_2024 = "FASE_24" if "FASE_24" in df_2024_uni.columns else None

# Aplica transformações específicas por ano
if col_fase_2022:
    df_2022_trans = aplicar_transformacoes_fase_turma(df_2022_uni, col_fase_2022, ano=2022)
else:
    df_2022_trans = df_2022_uni.copy()

if col_fase_2023:
    df_2023_trans = aplicar_transformacoes_fase_turma(df_2023_uni, col_fase_2023, ano=2023)
else:
    df_2023_trans = df_2023_uni.copy()
    
if col_fase_2024:
    df_2024_trans = aplicar_transformacoes_fase_turma(df_2024_uni, col_fase_2024, ano=2024)
else:
    df_2024_trans = df_2024_uni.copy()

print("✅ Transformações de FASE e TURMA aplicadas")
if col_fase_2022:
    print(f"\nDistribuição de FASE_PADRONIZADA (2022):")
    print(df_2022_trans["FASE_PADRONIZADA"].value_counts())

---

## 🏗️ Etapa 5: Criação de Features Derivadas

Cria flags e variáveis calculadas a partir dos dados brutos.

In [ ]:
# Cria coluna VETERANO (ingressou antes de 2022)
if "ANO_INGRESSO_22" in df_2022_trans.columns:
    df_2022_feat= criar_coluna_veterano(df_2022_trans, "ANO_INGRESSO_22", ano_corte=2022)
else:
    df_2022_feat = df_2022_trans.copy()

# Cria coluna EM_FASE (está na fase ideal)
if "FASE_PADRONIZADA" in df_2022_feat.columns and "FASE_IDEAL_22" in df_2022_feat.columns:
    df_2022_feat = criar_coluna_em_fase(df_2022_feat, "FASE_PADRONIZADA", "FASE_IDEAL_22")

print("✅ Features derivadas criadas")
if "VETERANO" in df_2022_feat.columns:
    print(f"\nDistribuição VETERANO:")
    print(df_2022_feat["VETERANO"].value_counts(normalize=True) * 100)

---

## 🔗 Etapa 6: Consolidação dos DataFrames

Merge dos 3 anos usando NOME como chave.

In [ ]:
# Consolida DataFrames
df_consolidado = consolidar_dataframes(
    [df_2022_feat, df_2023_trans, df_2024_trans],
    id_col="NOME",
    sufixos=["_2022", "_2023", "_2024"]
)

print(f"✅ Consolidação concluída")
print(f"  Dimensão final: {df_consolidado.shape[0]} linhas × {df_consolidado.shape[1]} colunas")
print(f"  Alunos únicos: {df_consolidado['NOME'].nunique()}")

---

## 📊 Etapa 7: Análise de Qualidade

Diagnóstico de valores ausentes e tipos de dados.

In [ ]:
# Análise de nulos
relatorio_nulos = analise_nulos(df_consolidado)

print("📊 Top 10 colunas com mais valores ausentes:\n")
print(relatorio_nulos.head(10))

# Estatísticas gerais
print(f"\n📈 Estatísticas de qualidade:")
print(f"  Total de colunas: {len(df_consolidado.columns)}")
print(f"  Colunas com nulos: {len(relatorio_nulos)}")
print(f"  % médio de nulos (colunas afetadas): {relatorio_nulos['perc_nulos'].mean():.2f}%")

---

## 💾 Etapa 8: Exportação

Salva dataset consolidado para uso posterior.

In [ ]:
# Define caminho de saída 
output_dir = project_root / "app" / "data" / "processed"
output_dir.mkdir(exist_ok=True, parents=True)

output_file = output_dir / "dataset_consolidado_2022_2024.parquet"

# Salva em formato Parquet (mais eficiente que CSV)
df_consolidado.to_parquet(output_file, index=False)

print(f"✅ Dataset consolidado salvo em:")
print(f"  {output_file}")
print(f"\n📦 Tamanho do arquivo: {output_file.stat().st_size / 1024 / 1024:.2f} MB")

---

## ✅ Conclusão

### Entregáveis
- ✅ Dataset consolidado de 2022-2024
- ✅ Features padronizadas e normalizadas
- ✅ Variáveis derivadas criadas (VETERANO, EM_FASE)
- ✅ Relatório de qualidade de dados

### Próximos Passos
1. **EDA detalhada** → `eda_passos_magicos.ipynb`
2. **Feature Engineering** → Engenharia avançada de features
3. **Modelagem** → Treinamento de modelos preditivos

### Referências
- Funções: `scripts/data_processing.py`, `scripts/notebook_feature_engineering.py`
- Testes: `tests/test_data_processing.py`, `tests/test_notebook_feature_engineering.py`
- Documentação: [README.md](../README.md)
